# get familiar with gradio as a go to tool for prototyping AI apps

In [1]:
import gradio as gr

In [2]:
def greet(name):
    return "Hello  " + name + "!"

In [3]:
demo = gr.Interface(fn=greet, inputs="text", outputs="text")

In [4]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
demo.close()

Closing server running on port: 7860


In [6]:
gr.close_all()

# add custom input and output components like Textbox

In [7]:
message_input = gr.Textbox(label="Your message: ", info="Enter a message for gpt-4o-mini", lines=7)
message_output = gr.Textbox(label="Response: ", info="Reply from gpt-4o-mini:", lines=7)
examples = ["hello", "howdy"]

ui_interface = gr.Interface(
    fn=greet,
    title="Greetings",
    inputs=[message_input],
    outputs=[message_output],
    examples=examples,
    flagging_mode="never"
)

ui_interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
ui_interface.close()

Closing server running on port: 7860


# write call to llm 

In [9]:
from openai import OpenAI
from dotenv import load_dotenv

In [62]:
openai_client = OpenAI() 

In [10]:
load_dotenv()

True

In [11]:
def llm_call(user_message):
    try:
        openai_client = OpenAI()
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "you are an helpful assistant"
                },
                {
                    "role": "user",
                    "content": user_message
                }
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error occurred while calling OpenAI LLM: {e}")

In [12]:
llm_call("hi")

'Hello! How can I assist you today?'

# now wrap this in simple gradio ui

In [13]:
llm_call_ui = gr.Interface(
    fn=llm_call, 
    title="LLM Call",
    inputs=gr.Textbox(label="Your Message: ", info="Enter message for gpt-4o-mini", lines=7),
    outputs=gr.Textbox(label="AI Response: ", info="Message from gpt-4o-mini", lines=7),
    examples=["Hi There!", "What is the capital of France?"]
)

In [14]:
llm_call_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
llm_call_ui.close()

Closing server running on port: 7860


In [16]:
gr.close_all()

# make llm respond in markdown

In [17]:
def llm_call_md(user_message):
    try:
        openai_client = OpenAI()
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "you are an helpful assistant. Repond strictly in Markdown format."
                },
                {
                    "role": "user",
                    "content": user_message
                }
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error occurred while calling OpenAI LLM: {e}")

In [21]:
llm_md_ui = gr.Interface(
    fn=llm_call_md,
    title="Markdown LLM Call",
    inputs=[gr.Textbox(label="Your message:", info="Enter message for LLM", lines=7)],
    outputs=[gr.Markdown(label="AI Response: ")],
    examples=["Write 100 words essay on llms", "Explain MLOPS to a 10 years old in 2 paragraphs"],
    flagging_mode="never"
)

In [22]:
llm_md_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [23]:
gr.close_all()

Closing server running on port: 7860


# make them stream

In [24]:
def llm_call_md_stream(user_message):
    try:
        openai_client = OpenAI()
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "you are an helpful assistant. Repond strictly in Markdown format."
                },
                {
                    "role": "user",
                    "content": user_message
                }
            ],
            stream=True
        )
        
        result = ""
        for chunk in response:
            if chunk.choices[0].delta.content is not None:
                result = result + chunk.choices[0].delta.content
                yield result
                
    except Exception as e:
        print(f"Error occurred while calling OpenAI LLM: {e}")
        yield f"⚠️ Error: {e}"

In [28]:
llm_md_stream_ui = gr.Interface(
    fn=llm_call_md_stream,
    title="Markdown LLM Call",
    inputs=[gr.Textbox(label="Your message:", info="Enter message for LLM", lines=7)],
    outputs=[gr.Markdown(label="AI Response: ")],
    examples=["What is the Capital of France?", "Please Explain Transformer architecture under 500 words"],
    flagging_mode="never"
)

In [29]:
llm_md_stream_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [30]:
gr.close_all()

Closing server running on port: 7860


# add dropdown menu to select a value

In [31]:
def distributor(user_message, llm_model):
    if llm_model == "GPT":
        yield from llm_call_md_stream(user_message)
    if llm_model == "Claude":
        yield from llm_call_md_stream(user_message)
    

In [32]:
dd = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
examples = [
    ["Explain LLM like explaining to a 10 years old", "GPT"],
    ["Explain Transformers like explaining to a 10 years old", "Claude"]
]

In [33]:
gr_ui = gr.Interface(
    fn=distributor,
    inputs=[gr.Textbox(label="Your message: ", info="Please enter message for llm model", lines=7), dd],
    outputs=gr.Markdown(label="Response"),
    examples=examples,
    flagging_mode="never",
    title="Textbox + Dropdown implementation"
)

In [34]:
gr_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
gr.close_all()

# lets convert our brochure generator into gradio app!

1. import all the code, remove print statements and only keep stream response

In [35]:
import os
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_all_links(url):
    try:
        # Send an HTTP GET request with a User-Agent to avoid being blocked
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        
        # Raise an exception for bad status codes (404, 500, etc.)
        response.raise_for_status()
        
        # Parse the HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        links = []
        # Find all <a> tags with an 'href' attribute
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            # Convert relative URLs (e.g., "/about") to absolute URLs
            full_url = urljoin(url, href)
            links.append(full_url)
            
        return links

    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return []


In [36]:
from pydantic import BaseModel

class FilteredLinksBatch(BaseModel):
    useful_links: list[str]

In [55]:
def filter_links_with_llm(company_url, all_links):
    """
    Uses OpenAI to filter a raw list of URLs down to only the ones
    relevant for generating a corporate brochure.
    """
    
    prompt = f"""
    You are an AI assistant helping build a company brochure for {company_url}.
    Review the following list of URLs extracted from the company's website.
    Identify which links are highly relevant for gathering foundational corporate information, 
    such as company history, team, mission, products, services, solutions, case studies, or contact details.
    
    Exclude irrelevant links like social media sharing buttons (twitter, facebook, linkedin), 
    login pages, privacy policies, terms of service, RSS feeds, or duplicate anchor tags.
    
    Return a clean list containing ONLY the useful absolute URLs.
    
    List of URLs to analyze:
    {json.dumps(all_links, indent=2)}
    """   
    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a professional web data analyst."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        response_format=FilteredLinksBatch
    )
    useful_links = response.choices[0].message.parsed.useful_links
    print(f"Found {len(useful_links)} useful links")
    return useful_links

In [56]:
import requests
from bs4 import BeautifulSoup

def scrape_content_from_links(links):
    """
    Loops through the filtered list of URLs, extracts the clean text 
    from each webpage, and combines them into one raw knowledge base.
    """
    combined_context = ""
    
    for link in links:
        try:
            print(f"Scraping content from: {link}")
            
            # Send GET request with a generic user agent to prevent basic bot blocks
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
            response = requests.get(link, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Remove scripts and style elements so we don't capture code or CSS
            for script_or_style in soup(["script", "style", "meta", "noscript"]):
                script_or_style.decompose()
                
            # Extract plain text from the webpage HTML
            text = soup.get_text(separator=' ')
            
            # Clean up whitespace, tabs, and newline gaps
            clean_text = " ".join(text.split())
            
            # Append to our master context with a clear header for the LLM
            combined_context += f"\n\n--- Source URL: {link} ---\n{clean_text}"
            
        except Exception as e:
            print(f"  [Error] Skipping {link}: {e}")
            
    return combined_context

In [57]:
def generate_brochure(company_url, knowledge_base):
    """
    Sends the compiled web text data to OpenAI to generate 
    a highly structured, elegant Markdown corporate brochure.
    """
    
    system_instruction = (
        "You are an elite corporate copywriter and marketing director. "
        "Your task is to generate a comprehensive, highly polished, and compelling "
        "corporate brochure based strictly on the provided context. Output the response "
        "in clean, perfectly formatted Markdown with an impact-driven layout."
    )
    
    user_prompt = f"""
    Create a detailed, publication-ready corporate brochure for the company at {company_url}.
    Use the raw webpage content below as your single source of truth. Extract key elements, 
    organize them logically, and make it engaging.
    
    Structure the brochure with the following sections:
    - **Header**: Company name, tagline, and an impact statement.
    - **Executive Summary / About Us**: Corporate identity, mission, vision, and core principles.
    - **Products, Services & Solutions**: Detailed breakdown of their main offerings and unique value propositions.
    - **Why Choose Us**: Key differentiators, achievements, or strengths extracted from the text.
    - **Contact Information**: Clear summary of addresses, key phone numbers, or corporate contact info.
    
    Raw context data:
    {knowledge_base}
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": system_instruction
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.7,
        stream=True
    )
    partial_text = ""
    for chunk in response:
        if chunk.choices[0].delta.content is not None:
            partial_text += chunk.choices[0].delta.content
            yield partial_text

In [58]:
def main(company_url):
    all_links = get_all_links(company_url)
    useful_links = filter_links_with_llm(company_url=company_url, all_links=all_links)
    raw_corporate_knowledge = scrape_content_from_links(useful_links)
    for text_update in generate_brochure(company_url, raw_corporate_knowledge):
        yield text_update

In [63]:
brochure_generator_stream_ui = gr.Interface(
    fn=main,
    title="Brochure Generator",
    inputs=[gr.Textbox(label="Company url:", info="Enter company website link for generating brochure", lines=1)],
    outputs=[gr.Markdown(label="Generated Brochure : ")],
    examples=["https://www.bmw.in/en/index.html", "https://www.toyotabharat.com/"],
    flagging_mode="never"
)

In [64]:
brochure_generator_stream_ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Found 31 useful links
Scraping content from: https://www.toyotabharat.com/find-a-dealer/
Scraping content from: https://www.toyotabharat.com/testdrive/
Scraping content from: https://www.toyotabharat.com/pricelist/
Scraping content from: https://www.toyotabharat.com/brochure/
Scraping content from: https://www.toyotabharat.com/buynow/?exchange=1
Scraping content from: https://www.toyotabharat.com/toyota-in-india/happier-path/
Scraping content from: https://www.toyotabharat.com/showroom/glanza/
Scraping content from: https://www.toyotabharat.com/showroom/urbancruiser-taisor/
Scraping content from: https://www.toyotabharat.com/showroom/rumion/
Scraping content from: https://www.toyotabharat.com/showroom/urbancruiser-hyryder/
Scraping content from: https://www.toyotabharat.com/showroom/urbancruiser-ebella/
Scraping content from: https://www.toyotabharat.com/showroom/innova-crysta/
Scraping content from: https://www.toyotabharat.com/showroom/innova/
Scraping content from: https://www.toyot

In [61]:
gr.close_all()

Closing server running on port: 7861
Closing server running on port: 7860
